# SAADE–AUA v9.1 · ПОВНА корекція AASIST/AASIST-L до repeat-padding

**Призначення:** виправити вхідний акустичний протокол AASIST та AASIST-L **для всього відкритого AUA і всіх публічних REAL–TTS пар**. Ні навчання, ні скорочення вибірки. Усі 21 сценарій залишаються **бітово ідентичними v9 до етапу padding**.

- AASIST/AASIST-L: заново інференс із `repeat(valid_audio)[:64600]`, заново пороги з **усіх 110 train-пар**, повні 21 сценарій.
- Wav2Vec2: **жодного нового інференсу**; її 292 110 прогнозів переносимо з попереднього завершеного v9 і перевіряємо контрольні суми/поріг/покриття.
- Нові результати в окремому каталозі `SAADE_AUA_2026_v9_1_AASIST_REPEAT_PADDING`, старі v9 не змінюються.
- `hidden_test`: **не читаємо й не оцінюємо**. Корекція є зміною протоколу після validation, тому наукові результати потрібно чітко позначати як post-hoc preprocessing correction.

**Важливо:** `RUN ALL` запустить повний багатогодинний тест двох ONNX-моделей; SQLite-кеш дозволяє відновити його після переривання.

## 0. Конфігурація — повний експеримент без лімітів

In [ ]:
from pathlib import Path
PROJECT = Path('/home/jovyan/work/audio_real_fake')
AUA = PROJECT/'AUA'                 # READ ONLY
RESULTS_ROOT = PROJECT/'results'    # ONLY OUTPUT ROOT

RUN_PHASE = 'FULL'                  # ONLY FULL; hidden intentionally inaccessible
MODEL_ENABLED = {'AASIST_ONNX': True, 'AASIST_L_ONNX': True, 'WAV2VEC2_STAFFORD': True}
INCLUDE_ALL_AUA = True             # УСІ придатні канонічні аудіофайли
INCLUDE_ALL_PUBLIC_MATCHED = True  # УСІ пари train/adaptation/validation; hidden окремо
ALL_NONOVERLAPPING_SEGMENTS = True # усі 4-секундні вікна, не тільки початок WAV
MIN_TAIL_SEC = 0.5                 # коротший хвіст запису не є надійним аудіотестом
VERIFY_SHA256 = True
BOOTSTRAP_REPS = 2000
SEED = 20261116
INSTALL_MISSING = False            # якщо потрібні залежності — увімкнути True один раз
SHOW_TRAIN_AUDIO_EXAMPLE = True
CACHE_MP3_TRANSFORMS = True
# Для контрольованого FIRST RUN прихований набір НІКОЛИ не відкривається автоматично.
# Hidden / FREEZE відсутні у цьому корекційному ноутбуці
# Вхідні дані та сценарії тотожні v9; змінюємо лише ONNX padding.
REUSE_V9_WAV2VEC2 = True

## 1. Середовище, SHA-256 і відокремлені каталоги результатів

In [ ]:
import os, sys, io, gc, json, math, time, zipfile, hashlib, shutil, subprocess, tempfile, urllib.request, warnings, itertools, sqlite3, types
from fractions import Fraction
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display, Audio

assert RUN_PHASE=='FULL', 'Цей ноутбук виправляє лише FULL, hidden закрито'
assert AUA.is_dir(), f'AUA не знайдено: {AUA}'
assert PROJECT.is_dir(), f'Не знайдено проєкт: {PROJECT}'
RESULTS_ROOT.mkdir(parents=True,exist_ok=True)
OLD_BASE=RESULTS_ROOT/'SAADE_AUA_2026_v9_COMPLETE_EVALUATION'
OLD_OUT=OLD_BASE/'FULL_EXPERIMENT'
assert OLD_OUT.is_dir(),f'Не знайдено завершений v9: {OLD_OUT}'
BASE=RESULTS_ROOT/'SAADE_AUA_2026_v9_1_AASIST_REPEAT_PADDING'
BASE.mkdir(parents=True,exist_ok=True)
assert AUA.resolve() not in BASE.resolve().parents and 'melnyk' not in str(BASE).lower()
MODEL_CACHE=OLD_BASE/'pretrained_model_cache'  # READ ONLY: вже завантажені перевірені ONNX
assert MODEL_CACHE.is_dir(), 'Немає кешу pretrained-моделей v9'
OUT=BASE/'FULL_EXPERIMENT';OUT.mkdir(exist_ok=True)
V6_CANDIDATES=[RESULTS_ROOT/'SAADE_AUA_2026_v6_2_MATCHED_TTS',RESULTS_ROOT/'SAADE_AUA_2026_v6_MATCHED_TTS']
V6_DIR=next((p for p in V6_CANDIDATES if (p/'V6_LOCKED_MATCHED_SPLITS.csv').exists()),None)
assert V6_DIR is not None,'Не знайдено V6_LOCKED_MATCHED_SPLITS.csv: REAL–TTS підкорпус відсутній'
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), 'Потрібен ffmpeg/ffprobe (для всіх форматів та MP3)'
if INSTALL_MISSING:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','numpy','pandas','scipy','soundfile','scikit-learn',
         'matplotlib','torch','transformers>=4.40,<5','onnxruntime','huggingface_hub','statsmodels','safetensors'])
import soundfile as sf
from scipy import signal,stats
from sklearn.metrics import balanced_accuracy_score,roc_auc_score,roc_curve,f1_score,confusion_matrix
import matplotlib.pyplot as plt
SR=16000;WINDOW_SEC=4.0;WINDOW_N=int(SR*WINDOW_SEC)
PROTOCOL_ID='SAADE-v9-20261116-full-canonical-all-segments-21-v8-compatible'  # НЕ МІНЯТИ: незмінні noise/IR seeds
PREPROCESSING_ID='SAADE-v9.1-AASIST-repeat-active-audio-to-64600'
OLD_CONFIG=json.loads((OLD_OUT/'00_COMPLETE_PROTOCOL.json').read_text(encoding='utf8'))
OLD_PROVENANCE=json.loads((OLD_OUT/'03_MODEL_PROVENANCE.json').read_text(encoding='utf8'))
OLD_CALIBRATION=json.loads((OLD_OUT/'04_CALIBRATION.json').read_text(encoding='utf8'))
OLD_READINESS=json.loads((OLD_OUT/'10_FULL_EXPERIMENT_READINESS.json').read_text(encoding='utf8'))
assert OLD_CONFIG['protocol']==PROTOCOL_ID and OLD_READINESS['actual_predictions']==876330
assert OLD_READINESS['hidden_test_opened'] is False
assert not (OLD_BASE/'HIDDEN_ONCE_STARTED_v9.json').exists(), 'v9 hidden вже починався — зупинено'
assert not (OLD_BASE/'FROZEN_PROTOCOL_v9.json').exists(), 'v9 уже заморожений — корекція потребує нового протоколу'

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()

def normhash(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()
def write_json_atomic(path,obj):
    tmp=Path(str(path)+'.part');tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf8');os.replace(tmp,path)
def stable_seed(*parts):return int.from_bytes(hashlib.sha256('||'.join(map(str,parts)).encode()).digest()[:8],'big')%(2**32)
def audio_duration(path):
    try:
        x=sf.info(str(path));return float(x.duration)
    except Exception:
        proc=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)],
                            capture_output=True,text=True,check=True,timeout=60)
        return float(proc.stdout.strip())
def read_wave(path,start_sec=0,active_n=WINDOW_N):
    active_n=int(active_n)
    if active_n<1 or active_n>WINDOW_N:raise ValueError('Некоректна активна довжина')
    try:
        with sf.SoundFile(str(path)) as f:
            rate=f.samplerate
            f.seek(max(0,int(round(float(start_sec)*rate))))
            wave=f.read(frames=int(math.ceil(WINDOW_SEC*rate)),dtype='float32',always_2d=True).mean(axis=1)
        if rate!=SR:
            p=Fraction(SR,rate);wave=signal.resample_poly(wave,p.numerator,p.denominator)
    except Exception:
        args=['ffmpeg','-nostdin','-v','error','-ss',str(max(0,float(start_sec))),'-i',str(path),'-t',str(WINDOW_SEC),
              '-ac','1','-ar',str(SR),'-f','f32le','-acodec','pcm_f32le','pipe:1']
        proc=subprocess.run(args,capture_output=True,timeout=180,check=True)
        wave=np.frombuffer(proc.stdout,dtype='<f4').copy()
    wave=np.nan_to_num(np.asarray(wave,dtype=np.float32),nan=0.,posinf=0.,neginf=0.)
    if not len(wave):raise ValueError('Порожнє декодоване аудіо: '+str(path))
    wave=wave[:active_n]
    return np.pad(wave,(0,max(0,WINDOW_N-len(wave))))[:WINDOW_N].astype(np.float32)

print('CORRECTED FULL MODE / без обмежень:',INCLUDE_ALL_AUA,INCLUDE_ALL_PUBLIC_MATCHED,ALL_NONOVERLAPPING_SEGMENTS)
print('Дані:',AUA,'| REAL–TTS:',V6_DIR,'| Результати:',OUT)

## 2. Каталог AUA з авторитетних метаданих (тільки читання)

In [ ]:
_EMBEDDED_DATA_ADAPTER = '\n"""SAADE v3 — AUA authoritative metadata adapter and conservative parent splits.\n\nThis module reads *only* canonical data under AUA/01_audio and AUA source metadata.\nIt never edits the dataset. Audio provenance and human-speaker identity are distinct:\nS1108/S1109 with source_group_unknown are NOT human/synthetic voice IDs.\nUnknown-identity corpora are train-only in the default documented pilot protocol.\nNo script claims that a class label is independently expert verified.\n"""\nfrom __future__ import annotations\nfrom pathlib import Path\nfrom collections import defaultdict\nfrom dataclasses import replace\nimport hashlib, json, re, warnings\nimport numpy as np\nimport pandas as pd\nimport saade_final_core as C\n\nBAD_VALUES={\'\',\'nan\',\'none\',\'null\',\'unknown\',\'n/a\',\'na\'}\n\ndef clean(v):\n    if v is None: return \'\'\n    s=str(v).strip()\n    return \'\' if s.lower() in BAD_VALUES else s\n\ndef get(row,*cols):\n    for col in cols:\n        if col in row and clean(row[col]): return clean(row[col])\n    return \'\'\n\ndef read_csv(p):\n    p=Path(p)\n    if not p.is_file(): return pd.DataFrame()\n    return pd.read_csv(p,dtype=str,keep_default_na=False,low_memory=False)\n\ndef key_path(s): return str(s).replace(\'\\\\\',\'/\').strip().lstrip(\'./\')\n\ndef table_by(df,col):\n    d=defaultdict(list)\n    if col not in df: return d\n    for _,r in df.iterrows():\n        k=key_path(clean(r[col]))\n        if k: d[k].append(r.to_dict())\n    return d\n\ndef one_entry(mapping,key):\n    rows=mapping.get(key,[])\n    return rows[0] if len(rows)==1 else {}\n\ndef csv_source_candidates(root, patterns):\n    """Find potential original metadata without assuming every file describes speakers."""\n    out=[]\n    for sub in patterns:\n        folder=root/\'00_sources_and_licenses\'/\'audio\'/sub\n        if folder.is_dir():\n            out.extend([str(p) for p in folder.rglob(\'*\') if p.suffix.lower() in (\'.csv\',\'.tsv\',\'.json\',\'.jsonl\',\'.parquet\')])\n    return sorted(out)\n\ndef source_key(src_value,path):\n    s=(str(src_value)+\' \'+str(path)).lower()\n    if \'common_voice\' in s or \'common voice\' in s: return \'CommonVoice\'\n    if \'fleurs\' in s: return \'FLEURS\'\n    if \'drssu\' in s: return \'DRSSU\'\n    if \'new-tts\' in s or \'new_tts\' in s: return \'AUA_new_TTS\'\n    if \'/own_processed/\' in str(path).lower().replace(\'\\\\\',\'/\'): return \'AUA_own\'\n    return \'unknown_source\'\n\ndef valid_speaker(speaker,basis,source):\n    """Do not treat synthetic/technical group labels as verified speaker identity."""\n    basis=basis.lower()\n    if not speaker: return False\n    if any(t in basis for t in (\'unknown\',\'source_group\',\'fallback\',\'synthetic_collection\',\'batch\')):return False\n    if source in (\'DRSSU\',\'FLEURS\') and not basis:return False\n    if source==\'CommonVoice\': return \'client_id\' in basis\n    if source==\'AUA_own\': return True\n    return bool(\'speaker\' in basis or \'identity\' in basis or \'verified\' in basis)\n\ndef build_verified_catalog(cfg, require_source_metadata=True):\n    root=Path(cfg.root).expanduser().resolve();aud=root/\'01_audio\';out=Path(cfg.out)\n    if out.resolve()==root or root in out.resolve().parents:\n        raise ValueError(\'Output folder must not be placed inside AUA\')\n    if any(\'melnyk\' in part.casefold() for part in out.resolve().parts):\n        raise ValueError(\'Do not write to Melnyk workspace\')\n    out.mkdir(parents=True,exist_ok=True)\n    if not aud.is_dir():raise FileNotFoundError(f\'Cannot find {aud}\')\n    general=read_csv(aud/\'metadata_audio.csv\')\n    own=read_csv(aud/\'own_processed\'/\'metadata_audio.csv\')\n    tts=read_csv(aud/\'synthetic\'/\'new-tts\'/\'metadata_new-tts.csv\')\n    drssu=read_csv(root/\'00_sources_and_licenses\'/\'audio\'/\'drssu\'/\'metadata.csv\')\n    # Direct metadata matches first; basename match only when globally unambiguous.\n    by_work=table_by(general,\'working_path\');by_name=table_by(general,\'file_name\')\n    own_by_name=table_by(own,\'file_name\');own_by_work=table_by(own,\'working_path\')\n    tts_by_filename=table_by(tts,\'filename\')\n    # The AUA field source_record_id is `drssu:<selection_rank>`; the\n    # source CSV\'s `file_name` instead contains `generated/4308.wav`, etc.\n    # Join on selection_rank and verify source SHA, never on the folder name.\n    drssu_rank=table_by(drssu,\'selection_rank\')\n    # Report potential source speaker metadata for manual linkage without inventing IDs.\n    candidates=csv_source_candidates(root,[\'fleurs\',\'common_voice\',\'drssu\'])\n    C.save_json(out/\'00_available_metadata_sources.json\',dict(root=str(root),\n        general_rows=len(general),own_rows=len(own),tts_rows=len(tts),drssu_rows=len(drssu),candidate_files=candidates[:300]))\n    allowed=[aud/\'own_processed\',aud/\'external_selected\',aud/\'synthetic\']\n    files=sorted({p for d in allowed if d.is_dir() for p in d.rglob(\'*\')\n                  if p.is_file() and p.suffix.lower() in C.AUDIO_EXT})\n    rows=[]\n    for p in files:\n        rp=p.relative_to(root).as_posix()\n        name=p.name\n        g=one_entry(by_work,key_path(rp)) or one_entry(by_name,name)\n        o=one_entry(own_by_work,key_path(rp)) or one_entry(own_by_name,name)\n        t=one_entry(tts_by_filename,name)\n        source=source_key(get(g,\'source_dataset\'),rp)\n        source_record=get(o,\'source_record_id\') or get(g,\'source_record_id\')\n        src_sha=get(o,\'source_sha256\') or get(g,\'source_sha256\')\n        working_sha=get(o,\'working_sha256\') or get(g,\'working_sha256\')\n        sp=get(o,\'speaker_id\') or get(g,\'speaker_id\')\n        sp_basis=get(g,\'speaker_id_basis\') or get(o,\'speaker_id_basis\')\n        mode=get(g,\'mode\').lower()\n        note=[]\n        dr={}\n        if source==\'DRSSU\':\n            # AUA metadata uses drssu:selection_rank, NOT a verified voice ID.\n            rank=source_record.split(\':\')[-1] if source_record else \'\'\n            dr=one_entry(drssu_rank,rank)\n            # Missing rank is never silently inferred from a file name.\n            if not dr:note.append(\'DRSSU_SOURCE_ROW_NOT_LINKED\')\n        label=None;label_basis=\'NOT_VERIFIED\'\n        if source==\'DRSSU\' and dr:\n            flag=get(dr,\'generated\').lower()\n            # Class comes from the source register, NOT from its filename.\n            if flag in (\'true\',\'1\',\'yes\'):\n                label=1;label_basis=\'DRSSU_source_metadata_generated_true\'\n            elif flag in (\'false\',\'0\',\'no\'):\n                label=0;label_basis=\'DRSSU_source_metadata_generated_false\'\n            else:note.append(\'DRSSU_GENERATED_FLAG_INVALID\')\n            # A missing or conflicting source SHA must not be treated as\n            # independently corroborated provenance. Refuse the class.\n            original_sha=get(dr,\'sha256\')\n            if not original_sha or not src_sha:\n                note.append(\'DRSSU_SOURCE_SHA_MISSING\')\n                label=None;label_basis=\'NOT_VERIFIED\'\n            elif original_sha.lower()!=src_sha.lower():\n                note.append(\'DRSSU_SOURCE_SHA_CONFLICT\')\n                label=None;label_basis=\'NOT_VERIFIED\'\n        elif source==\'AUA_new_TTS\' and t:\n            # A generator-specific register establishes that this is TTS even\n            # when voice_id is unavailable. Unknown voice remains TRAIN ONLY.\n            if get(t,\'model_name\'):\n                label=1\n                label_basis=(\'TTS_source_generation_register\' if get(t,\'voice_id\')\n                             else \'TTS_source_generation_register_voice_unknown\')\n                if not get(t,\'voice_id\'):\n                    note.append(\'TTS_VOICE_ID_UNKNOWN_TRAIN_ONLY\')\n            else:\n                note.append(\'TTS_MODEL_NAME_MISSING\')\n        elif source in (\'CommonVoice\',\'FLEURS\') and g:\n            label=0;label_basis=f\'{source}_dataset_origin_real_assumed\'\n        elif source==\'AUA_own\' and o:\n            if get(o,\'source_path\') or get(o,\'source_record_id\'):\n                label=0;label_basis=\'AUA_own_original_record_mapping\'\n            else:note.append(\'OWN_ORIGINAL_MAPPING_MISSING\')\n        else:\n            note.append(\'SOURCE_METADATA_MISSING\')\n        if source==\'unknown_source\':note.append(\'UNKNOWN_SOURCE\')\n        # Explicit mode only when it really denotes class, not READC etc.\n        class_mode={\'real\':0,\'human\':0,\'bona_fide\':0,\'genuine\':0,\n                    \'fake\':1,\'synthetic\':1,\'spoof\':1,\'tts\':1,\'vc\':1,\'generated\':1}.get(mode)\n        if class_mode is not None and label is not None and class_mode!=label:\n            note.append(\'EXPLICIT_MODE_CLASS_CONFLICT\')\n            label=None\n        known_speaker=valid_speaker(sp,sp_basis,source)\n        voice=get(t,\'voice_id\');generator=get(t,\'model_name\')\n        input_text_id=get(t,\'input_text_id\')\n        voice_confirmed=bool(source==\'AUA_new_TTS\' and voice and generator)\n        if source==\'DRSSU\':generator=\'UNKNOWN_DRSSU_GENERATOR\'\n        elif label==1 and not generator:generator=\'unknown_generator\'\n        if known_speaker:\n            group=f\'SPK::{source}::{sp}\'\n            confidence=\'metadata_speaker_verified_basis\' if source==\'CommonVoice\' else \'metadata_speaker\'\n            identity=\'speaker_confirmed\'\n            split_role=\'eligible\'\n        elif voice_confirmed:\n            group=f\'VOICE::{source}::{generator}::{voice}\'\n            confidence=\'metadata_voice\'\n            identity=\'synthetic_voice_confirmed\'\n            split_role=\'eligible\'\n        else:\n            # Conservative source-wise grouping, never pretend each DRSSU file is a new person.\n            # Unknown voices cannot enter validation/test under exploratory protocol.\n            group=f\'UNKNOWN_IDENTITY::{source}\'\n            confidence=\'source_identity_unknown\'\n            identity=\'unknown_identity\'\n            split_role=\'train_only\'\n            note.append(\'IDENTITY_UNKNOWN_TRAIN_ONLY\')\n        if source==\'DRSSU\' and not dr and require_source_metadata:label=None\n        duration=C.audio_info_seconds(p)\n        usable=(label in (0,1) and np.isfinite(duration) and duration>=float(cfg.min_seconds)\n                and not any(\'CONFLICT\' in s for s in note))\n        if not usable: note.append(\'UNUSABLE_OR_UNVERIFIED\')\n        rows.append(dict(path=str(p),relative_path=rp,source=source,source_dataset=get(g,\'source_dataset\') or source,\n             label=label,label_source=label_basis,label_conflict=any(\'CONFLICT\' in s for s in note),\n             speaker_id=sp if known_speaker else \'\',technical_speaker_id=sp,\n             speaker_id_basis=sp_basis,source_record_id=source_record,source_sha256=src_sha,\n             working_sha256=working_sha,group_id=group,group_confidence=confidence,\n             identity_status=identity,split_role=split_role,voice_id=voice,generator=generator,\n             input_text_id=input_text_id,transcription=get(o,\'transcription\') or get(g,\'transcription\'),\n             script_id=get(o,\'script_id\'),speaker_origin_group=get(o,\'speaker_origin_group\'),\n             script_profile=get(o,\'script_profile\'),dialect_authenticity=get(o,\'dialect_authenticity\'),\n             approval_status=get(o,\'approval_status\') or get(g,\'approval_status\') or get(t,\'approval_status\'),\n             license=get(g,\'license\') or get(t,\'license_terms\'),duration_sec=duration,\n             usable=bool(usable),warning=\'; \'.join(note),primary_meta_found=bool(g),\n             primary_source_meta_found=bool(dr) if source==\'DRSSU\' else bool(t) if source==\'AUA_new_TTS\' else bool(g)))\n    x=pd.DataFrame(rows)\n    if x.empty:raise ValueError(\'No audio found in canonical audio folders\')\n    # Dedup by exact working SHA, and by original source SHA within DRSSU.\n    # AUA contains 10 repeat DRSSU source records with distinct working WAV\n    # hashes; they must not be counted as 10 independent source recordings.\n    nonempty=x.working_sha256.fillna(\'\').ne(\'\')\n    dupl=nonempty & x.working_sha256.duplicated(keep=\'first\')\n    x.loc[dupl,\'usable\']=False\n    x.loc[dupl,\'warning\']=x.loc[dupl,\'warning\'].astype(str)+\'; DUPLICATE_WORKING_SHA\'\n    drmask=(x.source.eq(\'DRSSU\') & x.source_sha256.fillna(\'\').astype(str).str.strip().ne(\'\'))\n    dr_dup=drmask & x.loc[:, \'source_sha256\'].where(drmask).duplicated(keep=\'first\')\n    x.loc[dr_dup, \'usable\']=False\n    x.loc[dr_dup, \'warning\']=x.loc[dr_dup,\'warning\'].astype(str)+\'; DRSSU_DUPLICATE_SOURCE_SHA\'\n    # A SHA conflict is not proof of fake/real identity; keep it out of experiments.\n    x.to_csv(out/\'01_catalog_all.csv\',index=False)\n    x[~x.usable | x.warning.ne(\'\')].to_csv(out/\'01_catalog_issues.csv\',index=False)\n    summary=x.groupby([\'source\',\'label\',\'label_source\',\'identity_status\',\'split_role\'],dropna=False).agg(\n        files=(\'path\',\'size\'),hours=(\'duration_sec\',lambda s:pd.to_numeric(s,errors=\'coerce\').sum()/3600),\n        groups=(\'group_id\',\'nunique\'),usable=(\'usable\',\'sum\')).reset_index()\n    summary.to_csv(out/\'01_sources_summary.csv\',index=False)\n    crosstab=pd.crosstab(x.source,x.label,dropna=False)\n    crosstab.to_csv(out/\'01_source_class_table.csv\')\n    C.save_json(out/\'01_data_diagnostics.json\',dict(total=len(x),usable=int(x.usable.sum()),\n         class_labels_according_to_provenance=int(x.label.notna().sum()),\n         metadata_unlinked=int((~x.primary_meta_found).sum()),\n         unknown_identity_files=int(x.identity_status.eq(\'unknown_identity\').sum()),\n         label_source_counts=x.label_source.value_counts(dropna=False).to_dict(),\n         warning=\'Dataset origin real/fake is not expert audio validation; source-vs-class confounding may remain.\'))\n    dr=x[x.source.eq(\'DRSSU\')]\n    provenance=dict(\n        DRSSU_total=int(len(dr)),\n        DRSSU_linked=int(dr.primary_source_meta_found.sum()) if len(dr) else 0,\n        DRSSU_labelled_fake=int(dr.label.eq(1).sum()) if len(dr) else 0,\n        DRSSU_source_sha_conflicts=int(dr.warning.fillna(\'\').str.contains(\'DRSSU_SOURCE_SHA_CONFLICT\').sum()) if len(dr) else 0,\n        DRSSU_duplicate_source_sha=int(dr.warning.fillna(\'\').str.contains(\'DRSSU_DUPLICATE_SOURCE_SHA\').sum()) if len(dr) else 0,\n        DRSSU_split_role_train_only=int(dr.split_role.eq(\'train_only\').sum()) if len(dr) else 0,\n        warning=\'Known source sha duplicates and technical speaker identities are not independent data.\'\n    )\n    C.save_json(out/\'01_DRSSU_PROVENANCE.json\',provenance)\n    print(\'DRSSU provenance:\',provenance)\n    print(\'AUTHORITATIVE CATALOG:\',len(x),\'usable\',x.usable.sum())\n    print(summary.to_string(index=False,max_rows=20))\n    return x\n\n\ndef prepare_parent_splits(catalog,cfg,max_parent_files=850,unknown_train_cap=150,policy=\'exploratory\'):\n    """Create group-safe split on VERIFIED speaker/voice groups; append unknown-identity TRAIN ONLY.\n\n    A DRSSU `source_group_unknown` is NOT a valid independent synthetic voice ID.\n    Paired source/class prerequisite for strict publication mode.\n    """\n    out=Path(cfg.out);out.mkdir(parents=True,exist_ok=True)\n    v=catalog[catalog.usable & catalog.label.isin([0,1])].copy()\n    v[\'label\']=v.label.astype(int)\n    eligible=v[v.split_role.eq(\'eligible\')].copy()\n    trainonly=v[v.split_role.eq(\'train_only\')].copy()\n    if eligible.empty or eligible.label.nunique()!=2:\n        raise ValueError(\'Not enough metadata-confirmed REAL/FAKE groups for validation/test. Check TTS voice register.\')\n    src=pd.crosstab(v.source,v.label)\n    for val in (0,1):\n        if val not in src:src[val]=0\n    matched=src[(src[0]>0)&(src[1]>0)].index.tolist()\n    if policy==\'strict_source\' and not matched:\n        raise ValueError(\'STRICT SOURCE MATCH FAILED: no confirmed source contains both REAL and FAKE. \'\n                         \'Provide comparable real/fake source data; do not present exploratory as final.\')\n    # Strict mode uses matched sources only, including identity-known subset.\n    if policy==\'strict_source\':\n        eligible=eligible[eligible.source.isin(matched)].copy()\n        trainonly=trainonly.iloc[:0].copy()\n    cap=int(max_parent_files)\n    if cap>0:\n        # Set cap for split-eligible and train-only separately so unverified voices never dominate validation.\n        elig_cap=max(50,cap-int(unknown_train_cap))\n        local=replace(cfg,max_items=elig_cap)\n        eligible=C.get_experimental_subset(eligible,local)\n    # min groups for val/test from each label\n    cnt=eligible.groupby(\'label\').group_id.nunique()\n    if len(cnt)!=2 or cnt.min()<3:\n        raise ValueError(\'Need at least 3 independently identified speaker/voice groups PER CLASS\')\n    split=C.make_splits(eligible,cfg)\n    if not trainonly.empty and unknown_train_cap>0:\n        # Select per-source examples without granting speaker independence.\n        def order_key(z):\n            return z.sort_values([\'source\',\'source_record_id\',\'relative_path\'])\n        trainonly=order_key(trainonly)\n        chunks=[]\n        for _,a in trainonly.groupby(\'source\',sort=True):\n            chunks.append(a.head(max(1,int(unknown_train_cap)//max(1,trainonly.source.nunique()))))\n        trainonly=pd.concat(chunks,ignore_index=True)\n        trainonly[\'split\']=\'train\'\n        split=pd.concat([split,trainonly],ignore_index=True)\n    # Guard source and working hashes, speaker/voice group IDs\n    for col in (\'group_id\',\'source_sha256\',\'working_sha256\',\'path\'):\n        if col not in split:continue\n        z=split.copy()\n        z=z[z[col].fillna(\'\').astype(str).str.strip().ne(\'\')]\n        leaks=z.groupby(col).split.nunique()\n        if leaks.gt(1).any():\n            raise AssertionError(f\'Cross-split leakage for {col}: {int(leaks.gt(1).sum())} groups\')\n    split[\'parent_file_id\']=split.relative_path\n    split.to_csv(out/\'03_parent_splits.csv\',index=False)\n    # Cross-split TTS prompt reuse: diagnostic only; identical input text is NOT a verified voice leak.\n    if \'input_text_id\' in split.columns:\n        tx=split[split.input_text_id.fillna(\'\').astype(str).str.strip().ne(\'\')]\n        repeat=tx.groupby(\'input_text_id\').agg(\n            splits=(\'split\',lambda a:\' | \'.join(sorted(set(a)))),\n            n_splits=(\'split\',\'nunique\'),files=(\'path\',\'size\'),voices=(\'voice_id\',\'nunique\')).reset_index()\n        repeat.to_csv(out/\'03_TTS_TEXT_OVERLAP.csv\',index=False)\n        C.save_json(out/\'03_TTS_TEXT_OVERLAP_SUMMARY.json\',dict(\n            cross_split_text_ids=int((repeat.n_splits>1).sum()),\n            note=\'Cross-split TTS input texts can influence text-generalization. They are NOT automatically speaker leaks.\'))\n    report=(split.groupby([\'split\',\'source\',\'label\',\'identity_status\'],dropna=False)\n                .agg(parent_files=(\'path\',\'size\'),groups=(\'group_id\',\'nunique\'))\n                .reset_index())\n    report.to_csv(out/\'03_parent_split_summary.csv\',index=False)\n    C.save_json(out/\'03_split_provenance.json\',dict(policy=policy,matched_sources=matched,\n         unknown_voice_files_train_only=int(split.split_role.eq(\'train_only\').sum()),\n         train_only_unknown_identity=True,\n         warning=\'Group-safe split conditional on metadata. Does NOT resolve source/class confounding or hidden identity repetition.\'))\n    return split,report\n\n\ndef data_readiness_report(catalog, parent_splits, cfg, policy):\n    out=Path(cfg.out)\n    both=(pd.crosstab(catalog[catalog.usable].source,catalog[catalog.usable].label)\n          .reindex(columns=[0,1],fill_value=0))\n    pairs=both[(both[0]>0)&(both[1]>0)]\n    check=[\n        (\'provenance_source_links\',int((catalog.primary_meta_found).sum()),len(catalog),\'CHECK\' if (~catalog.primary_meta_found).any() else \'PASS\'),\n        (\'label_unverified\',int(catalog.label.isna().sum()),0,\'PASS\' if catalog.label.notna().all() else \'CHECK\'),\n        (\'unknown_identity_in_validation_test\',int(((parent_splits.split!=\'train\') & parent_splits.split_role.eq(\'train_only\')).sum()),0, \'PASS\' if not ((parent_splits.split!=\'train\') & parent_splits.split_role.eq(\'train_only\')).any() else \'FAIL\'),\n        (\'matched_source_both_classes\',len(pairs),1,\'PASS\' if len(pairs)>0 else \'FAIL\'),\n        (\'source_group_unknown_ID_reused_as_speaker\',int((parent_splits.identity_status.eq(\'unknown_identity\') & parent_splits.group_confidence.str.contains(\'metadata_speaker\')).sum()),0,\'PASS\'),\n        (\'source_class_confounding_resolved\',len(pairs),1,\'PASS\' if len(pairs)>0 else \'FAIL\')]\n    data=pd.DataFrame(check,columns=[\'item\',\'actual\',\'required\',\'status\'])\n    data.to_csv(out/\'DATA_READINESS.csv\',index=False)\n    C.save_json(out/\'DATA_READINESS.json\',dict(strict_ready=bool(len(pairs)>0 and policy==\'strict_source\'),\n        purpose=\'SAADE diagnostics and pilot ML; not expert evaluation\',\n        publication_warning=\'Matched source is necessary, not sufficient; capture-channel / speaker / text / generator confounds still require checks.\'))\n    return data\n'

# Авторитетний адаптер метаданих із SAADE v6.2, вбудований у цей ноутбук.
# Для нього надаємо мінімальні сумісні API, НЕ встановлюємо сторонні файли .py.
_core=types.ModuleType('saade_final_core')
_core.AUDIO_EXT={'.wav','.flac','.mp3','.ogg','.m4a','.aac'}
def _safe_audio_info_seconds(p):
    try:return audio_duration(p)
    except Exception:return float('nan')
_core.audio_info_seconds=_safe_audio_info_seconds
_core.save_json=lambda path,obj: write_json_atomic(path,obj)
sys.modules['saade_final_core']=_core
_data=types.ModuleType('saade_final_data')
exec(compile(_EMBEDDED_DATA_ADAPTER,'saade_final_data_embedded','exec'),_data.__dict__)
from types import SimpleNamespace
catalog_dir=OUT/'catalog';catalog_dir.mkdir(exist_ok=True)
catalog=_data.build_verified_catalog(SimpleNamespace(root=str(AUA),out=str(catalog_dir),min_seconds=0.5))
assert {'path','relative_path','source','label','usable','working_sha256','source_sha256'}.issubset(catalog.columns)
canonical=catalog.loc[catalog.usable & catalog.label.isin([0,1])].copy()
print(f'Усього в канонічному каталозі: {len(catalog)}; придатних: {len(canonical)}; пропущено {len(catalog)-len(canonical)} (із причинами в catalog/01_catalog_issues.csv)')
display(canonical.groupby(['source','label'],dropna=False).size().rename('recordings').reset_index())

## 3. Усі відкриті REAL–TTS пари; 69 hidden-пар залишаються ізольованими

In [ ]:
MANIFEST_PATH=V6_DIR/'V6_LOCKED_MATCHED_SPLITS.csv'
raw=pd.read_csv(MANIFEST_PATH,low_memory=False)
need={'path','pair_id','label','split','parent_file_id','tts_voice_id','group_id'}
assert need.issubset(raw),f'Немає колонок: {need-set(raw)}'
raw['label']=pd.to_numeric(raw.label,errors='raise').astype(int)
raw['pair_id']=raw.pair_id.astype(str)
assert raw.label.isin([0,1]).all()
assert set(raw.split.unique()).issubset({'train','adaptation','validation','hidden_test'})
assert raw.groupby('pair_id').split.nunique().max()==1,'Витік пари між split'
for col in ['parent_file_id','text_key','input_text_id','working_sha256']:
    if col in raw:
        sub=raw.dropna(subset=[col]).copy();sub=sub.loc[sub[col].astype(str).str.strip().ne('')]
        assert sub.groupby(col).split.nunique().max()<=1,f'Перетин split за {col}'
real=raw.loc[raw.label.eq(0)]
assert real.groupby('group_id').split.nunique().max()<=1,'Перетин людських голосових груп'
fake=raw.loc[raw.label.eq(1)]
assert fake.groupby('group_id').split.nunique().max()<=1,'Перетин ідентифікованих TTS-груп'
assert raw.groupby(['pair_id','label']).size().eq(1).all(),'Кожна пара має мати рівно REAL+FAKE'
assert raw.groupby('pair_id').label.nunique().eq(2).all()
public=raw.loc[raw.split.ne('hidden_test')].copy()
matched_groups=raw.loc[raw.label.eq(1)].groupby(['split','tts_voice_id']).pair_id.nunique().reset_index(name='pairs')
matched_groups.to_csv(OUT/'01_MATCHED_VOICE_SPLITS.csv',index=False)
# Hidden identities are only used to EXCLUDE any canonical AUA overlap, not to evaluate hidden WAV.
hidden=raw.loc[raw.split.eq('hidden_test')]
hidden_sha=set(hidden.working_sha256.dropna().astype(str).str.lower()) if 'working_sha256' in hidden else set()
hidden_sha={s for s in hidden_sha if len(s)==64}
hidden_paths={str(Path(p).resolve()) for p in hidden.path.astype(str)}

rec_path=V6_DIR/'V6_TTS_RECEIPTS.csv'
assert rec_path.exists(),'Немає V6_TTS_RECEIPTS.csv із SHA-256 синтезу'
receipts=pd.read_csv(rec_path)
rec_lookup={(str(r.pair_id),str(r.generated_path)):str(r.sha256).lower() for r in receipts.itertuples(index=False)}

pair_active={}
for pid,g in public.groupby('pair_id'):
    secs=[]
    for row in g.itertuples(index=False):
        assert Path(row.path).is_file(),f'Відсутній REAL–TTS WAV: {row.path}'
        start=float(getattr(row,'segment_start_sec',0) or 0)
        available=audio_duration(row.path)-start
        assert available>=MIN_TAIL_SEC,(pid,row.path,available)
        secs.append(available)
    pair_active[str(pid)]=int(min(WINDOW_N,max(1,math.floor(min(secs)*SR))))

matched=[]
for r in public.itertuples(index=False):
    expected=(rec_lookup.get((str(r.pair_id),str(r.path))) if int(r.label)==1
              else str(getattr(r,'working_sha256','')).lower())
    assert expected and len(expected)==64, f'Немає підтвердженої SHA для {r.pair_id} label={r.label}'
    matched.append(dict(sample_id=f'matched:{r.pair_id}:{int(r.label)}',record_id=f'matched:{r.pair_id}:{int(r.label)}',
       cohort='MATCHED_'+str(r.split).upper(),source='Piper_matched',split=str(r.split),
       pair_id=str(r.pair_id),label=int(r.label),path=str(r.path),start_sec=float(getattr(r,'segment_start_sec',0) or 0),
       active_n=pair_active[str(r.pair_id)],group_id=str(r.group_id),voice_id=str(r.tts_voice_id),
       expected_sha=expected,source_kind='matched_controlled'))
matched_df=pd.DataFrame(matched)
print('Усі публічні REAL–TTS пари (жодної підвибірки):')
display(matched_df.groupby(['cohort','label']).size().unstack(fill_value=0))
print('Приховані пари зарезервовано (НЕ тестуються в FULL):',hidden.pair_id.nunique())

## 4. Увесь канонічний AUA: усі неперекривні 4-секундні сегменти

In [ ]:
# Зберігаємо ВСІ канонічні записи, крім справді непридатних і пересічних із незалежним HIDDEN.
canon=canonical.copy()
canon['working_sha256']=canon.working_sha256.fillna('').astype(str).str.lower()
canon['hidden_overlap']=canon.working_sha256.isin(hidden_sha) | canon.path.astype(str).map(lambda p:str(Path(p).resolve()) in hidden_paths)
excluded_hidden=canon.loc[canon.hidden_overlap]
if len(excluded_hidden):
    print('Захист hidden: вилучено з відкритого AUA-тесту',len(excluded_hidden),'записів із тим самим SHA/шляхом')
canon=canon.loc[~canon.hidden_overlap].copy()
assert not canon.empty,'Немає придатних канонічних записів після контролю hidden'

aua_rows=[];too_short=[]
for r in canon.itertuples(index=False):
    duration=float(r.duration_sec)
    if not np.isfinite(duration) or duration<MIN_TAIL_SEC:
        too_short.append((r.relative_path,duration));continue
    # Усі неперекривні вікна [0,4),[4,8),...; останній хвіст включено, якщо >= MIN_TAIL_SEC.
    starts=np.arange(0,int(math.ceil(duration/WINDOW_SEC)),dtype=np.int64)*int(WINDOW_SEC)
    if not ALL_NONOVERLAPPING_SEGMENTS:starts=starts[:1]
    record_id='aua:'+hashlib.sha256(str(r.relative_path).encode()).hexdigest()[:20]
    for seg_i,start in enumerate(starts):
        sec=min(WINDOW_SEC,duration-float(start))
        if sec<MIN_TAIL_SEC:continue
        aua_rows.append(dict(sample_id=f'{record_id}:seg{seg_i:05d}',record_id=record_id,
           cohort='AUA_CANONICAL',source=str(r.source),split='source_inventory',pair_id='',
           label=int(r.label),path=str(r.path),start_sec=float(start),
           active_n=int(max(1,math.floor(sec*SR))),group_id=str(r.group_id),voice_id=str(r.voice_id),
           expected_sha=str(r.working_sha256),source_kind='original_AUA'))
aua_df=pd.DataFrame(aua_rows)
assert not aua_df.empty
assert aua_df.sample_id.is_unique
work=pd.concat([matched_df,aua_df],ignore_index=True) if INCLUDE_ALL_AUA else matched_df.copy()
assert work.sample_id.is_unique
work.to_csv(OUT/'01_PRIVATE_FULL_WORKLIST.csv',index=False) # НЕ включено у ZIP
coverage=work.groupby(['cohort','source','label']).agg(segments=('sample_id','size'),recordings=('record_id','nunique')).reset_index()
coverage.to_csv(OUT/'01_FULL_COVERAGE.csv',index=False)
print('Записів AUA:',aua_df.record_id.nunique(),'| 4-секундних вікон:',len(aua_df),'| matched вікон:',len(matched_df))
print('УСЬОГО заплановано інференсів:',len(work)*21*sum(MODEL_ENABLED.values()),'(21 сценарій × 3 моделі × всі вікна)')
display(coverage)

## 5. Ті самі 21 акустичний сценарій, що у v9

In [ ]:
SCENARIOS=[{'scenario':'clean','family':'clean','level':0,'parameter':None}]
for level,snr in enumerate([30,25,20,15,10],1):
    SCENARIOS.append({'scenario':f'noise_snr{snr}','family':'noise','level':level,'parameter':snr})
for level,kbps in enumerate([128,96,64,48,32],1):
    SCENARIOS.append({'scenario':f'mp3_{kbps}kbps','family':'mp3','level':level,'parameter':kbps})
for level,rt in enumerate([.15,.30,.45,.65,.85],1):
    SCENARIOS.append({'scenario':f'reverb_rt{rt:.2f}','family':'reverb','level':level,'parameter':rt})
for level,hz in enumerate([7500,6000,4500,3000,2000],1):
    SCENARIOS.append({'scenario':f'lowpass_{hz}Hz','family':'lowpass','level':level,'parameter':hz})
SC=pd.DataFrame(SCENARIOS)
assert len(SC)==21 and SC.scenario.is_unique
SC.to_csv(OUT/'02_SCENARIO_PROTOCOL.csv',index=False)
SCENARIO_HASH=normhash(SCENARIOS)

def mp3_roundtrip(active,kbps):
    with tempfile.TemporaryDirectory(prefix='saade_mp3_') as tmp:
        wd=Path(tmp); src=wd/'original.wav'; mp3=wd/'compressed.mp3'; dec=wd/'decoded.wav'
        sf.write(src,active,SR,subtype='PCM_16')
        args=['ffmpeg','-nostdin','-y','-hide_banner','-loglevel','error']
        subprocess.run(args+['-i',str(src),'-c:a','libmp3lame','-b:a',f'{kbps}k',
                   '-ar',str(SR),'-ac','1',str(mp3)],check=True,timeout=60,stderr=subprocess.PIPE)
        subprocess.run(args+['-i',str(mp3),'-ar',str(SR),'-ac','1',str(dec)],
                   check=True,timeout=60,stderr=subprocess.PIPE)
        y,sr=sf.read(dec,dtype='float32',always_2d=True)
        assert sr==SR
        y=y.mean(axis=1)
    return np.pad(y[:len(active)],(0,max(0,len(active)-len(y))))[:len(active)]

def apply_scenario(wave,sc,pair_id,active_n=None):
    """Perturb ONLY active audio, keep deterministic equal padding per REAL/FAKE pair."""
    n=WINDOW_N if active_n is None else int(active_n)
    if n<1 or n>WINDOW_N:raise ValueError('Bad active_n')
    active=wave[:n].copy()
    family=sc['family']; val=sc['parameter']
    if family=='clean':out=active
    elif family=='noise':
        # ОДНАКОВИЙ шумовий шаблон для REAL та FAKE однієї пари, SNR залежить від RMS кожного сигналу.
        rng=np.random.default_rng(stable_seed(PROTOCOL_ID,pair_id,sc['scenario']))
        z=rng.standard_normal(n).astype(np.float32)
        z/=np.sqrt(np.mean(z.astype(np.float64)**2)+1e-12)
        rms=np.sqrt(np.mean(active.astype(np.float64)**2)+1e-12)
        out=np.clip(active+z*(rms/(10**(float(val)/20))),-1,1)
    elif family=='mp3':out=mp3_roundtrip(active,int(val))
    elif family=='reverb':
        # Синтетичний IR, НЕ емпірично виміряний RT60 кімнати.
        rt=float(val);m=int(min(SR*1.5,SR*rt*1.2))
        t=np.arange(m)/SR;env=np.exp(-6.9078*t/rt)
        rng=np.random.default_rng(stable_seed(PROTOCOL_ID,'IR',sc['scenario']))
        h=rng.normal(size=m).astype(np.float32)*env
        h[0]+=4.0
        h/=np.sqrt(np.sum(h.astype(np.float64)**2)+1e-12)
        out=np.clip(signal.fftconvolve(active,h,mode='full')[:n],-1,1)
    elif family=='lowpass':
        sos=signal.butter(6,float(val),fs=SR,output='sos',btype='lowpass')
        out=signal.sosfilt(sos,active)
    else:raise KeyError(family)
    out=np.asarray(out,dtype=np.float32)
    assert out.shape==(n,) and np.isfinite(out).all()
    return np.pad(out,(0,WINDOW_N-n)).astype(np.float32)

def transform_cached(wave,sc,row,cache_root):
    """Cache deterministic MP3 only; cache fingerprint includes source SHA and scenario.
    This cache is never packaged into the review archive.
    """
    active_n=int(row.active_n)
    if not CACHE_MP3_TRANSFORMS or sc['family']!='mp3':
        return apply_scenario(wave,sc,str(row.pair_id if row.pair_id else row.sample_id),active_n)
    key=normhash({'protocol':PROTOCOL_ID,'source_sha':str(row.expected_sha),
                  'sample_id':str(row.sample_id),'label':int(row.label),
                  'scenario':sc['scenario'],'active_n':active_n})
    dest=Path(cache_root)/(key+'.npy')
    if dest.is_file():
        arr=np.load(dest,allow_pickle=False)
        if arr.shape!=(WINDOW_N,) or arr.dtype!=np.float32 or not np.isfinite(arr).all():
            raise RuntimeError('Пошкоджено кеш MP3: '+str(dest))
        return arr
    out=apply_scenario(wave,sc,str(row.pair_id if row.pair_id else row.sample_id),active_n)
    dest.parent.mkdir(parents=True,exist_ok=True)
    tmp=dest.with_name(dest.name+'.part.npy')
    np.save(tmp,out,allow_pickle=False)
    os.replace(tmp,dest)
    return out


probe=(.08*np.sin(2*np.pi*440*np.arange(WINDOW_N)/SR)).astype(np.float32)
for sc in SCENARIOS:
    z=apply_scenario(probe,sc,'unit-test',SR*3)
    assert z.shape==(WINDOW_N,) and np.isfinite(z).all() and not z[SR*3:].any()
assert np.array_equal(apply_scenario(probe,SCENARIOS[1],'A',SR),apply_scenario(probe,SCENARIOS[1],'A',SR))
print('21 сценарій: синтез, детермінованість, паддинг — OK')

## 6. Еталонне repeat-padding тільки для AASIST/AASIST-L (готові checkpoint)

In [ ]:
MODEL_SOURCES={
 'AASIST_ONNX':{'kind':'onnx','repo':'SpeechAntiSpoofingBenchmarks/AASIST',
                'file':'aasist.onnx','source':'https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST'},
 'AASIST_L_ONNX':{'kind':'onnx','repo':'SpeechAntiSpoofingBenchmarks/AASIST-L',
                  'file':'aasist-l.onnx','source':'https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST-L'},
 'WAV2VEC2_STAFFORD':{'kind':'hf','repo':'garystafford/wav2vec2-deepfake-voice-detector',
                     'expected_fake_index':1,
                     'source':'https://huggingface.co/garystafford/wav2vec2-deepfake-voice-detector'}
}
SELECTED_MODELS=[m for m,v in MODEL_ENABLED.items() if v]
if not SELECTED_MODELS:raise ValueError('Жодної моделі')
if not set(SELECTED_MODELS).issubset(MODEL_SOURCES):raise ValueError('Модель не має перевіреного адаптера')

def download_identity(repo,filename,dest,min_bytes=50000):
    dest=Path(dest);dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.is_file() and dest.stat().st_size>=min_bytes:return dest
    url=f'https://huggingface.co/{repo}/resolve/main/{filename}'
    req=urllib.request.Request(url,headers={'User-Agent':'SAADE-v8-research/1.0','Accept-Encoding':'identity'})
    part=dest.with_name(dest.name+'.part')
    try:
        with urllib.request.urlopen(req,timeout=180) as response, open(part,'wb') as f:
            while True:
                b=response.read(2*1024*1024)
                if not b:break
                f.write(b)
        if part.stat().st_size<min_bytes or part.open('rb').read(32).lstrip().lower().startswith(b'<html'):
            raise RuntimeError('Файл моделі не завантажено повністю: '+url)
        os.replace(part,dest)
    finally:part.unlink(missing_ok=True)
    return dest

def fake_index_from_labels(id2label,expected):
    labels={int(k):str(v).strip().lower() for k,v in id2label.items()}
    assert set(labels)=={0,1},'Не 2 класи: '+repr(labels)
    explicit=[k for k,v in labels.items() if 'fake' in v or 'spoof' in v or 'synthetic' in v]
    if len(explicit)==1 and explicit[0]!=expected:
        raise RuntimeError('Класи моделі не збігаються з документацією: '+str(labels))
    if len(explicit)==0 and set(labels.values())!={'label_0','label_1'}:
        raise RuntimeError('Неможливо підтвердити мітки FAKE/REAL: '+str(labels))
    if expected not in labels:raise ValueError('Неправильний FAKE index')
    return expected,labels

class OnnxAASIST:
    def __init__(self,name,cfg):
        import onnxruntime as ort
        self.name=name;self.source=cfg['source']
        fp=download_identity(cfg['repo'],cfg['file'],MODEL_CACHE/cfg['file'])
        self.weights_sha256=sha256_file(fp)
        self.model_revision=self.weights_sha256
        self.session=ort.InferenceSession(str(fp),providers=([ 'CUDAExecutionProvider','CPUExecutionProvider' ] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']))
        ins=self.session.get_inputs()
        assert len(ins)==1,'Непідтримуваний ONNX input'
        self.input_name=ins[0].name
        self.input_shape=ins[0].shape
        self.score_kind='negative_bonafide_logit (higher is fake, not probability)'
        self.labels={0:'spoof',1:'bonafide'}  # реальний порядок логітів, підтверджений аудитом ONNX ↔ PyTorch
    def score_fake(self,w,active_n=WINDOW_N):
        # Еталон AASIST: повторювати ДІЙСНИЙ активний сигнал до 64600 семплів.
        # Повторення zero-padding було б методологічно помилковим для коротких хвостів.
        n=int(active_n)
        assert 1 <= n <= WINDOW_N, f'Неправильна активна довжина: {n}'
        active=np.asarray(w[:n],dtype=np.float32)
        assert active.shape==(n,) and np.isfinite(active).all()
        x=np.tile(active, int(math.ceil(64600/n)))[:64600].astype(np.float32)
        assert x.shape==(64600,)
        sh=self.input_shape
        if len(sh)==2:inp=x[None,:]
        elif len(sh)==3 and sh[1] in (1,'1',None):inp=x[None,None,:]
        else:raise RuntimeError('Невідомий shape ONNX: '+str(sh))
        outs=self.session.run(None,{self.input_name:inp})
        vals=np.asarray(outs[0],dtype=float).reshape(-1)
        if vals.size==1:bonafide=vals[0]
        elif vals.size==2:bonafide=vals[1]
        else:raise RuntimeError('Непередбачений вихід AASIST: '+str(vals.shape))
        return float(-bonafide)
    def close(self):del self.session


def locate_hf_weights(repo,revision=None,fallback=None):
    """Find exact cached weights to fingerprint. No unknown revision is accepted."""
    candidates=[]
    try:
        from huggingface_hub import constants
        cache=Path(constants.HF_HUB_CACHE)
        root=cache/('models--'+repo.replace('/','--'))/'snapshots'
        if revision and revision!='unknown':
            q=root/revision/'model.safetensors'
            if q.exists():candidates.append(q)
        else:
            candidates.extend(root.glob('*/model.safetensors'))
    except Exception:pass
    if fallback and Path(fallback).exists():candidates.insert(0,Path(fallback))
    # Multiple ambiguous snapshots MUST NOT be chosen using mtime alone.
    hashes={sha256_file(p):p for p in candidates if p.is_file() and p.stat().st_size>50_000}
    if len(hashes)!=1:
        raise RuntimeError(f'Неможливо однозначно зафіксувати SHA-256 checkpoint {repo}. '
                           f'Знайдено варіантів: {len(hashes)}. Приберіть зайві версії кешу або '
                           'завантажте модель в окремий локальний каталог.')
    return next(iter(hashes)), str(next(iter(hashes.values())))

class HFWav2Vec2:
    def __init__(self,name,cfg):
        import torch
        from transformers import AutoFeatureExtractor,AutoModelForAudioClassification
        self.name=name;self.source=cfg['source'];self.torch=torch
        self.device='cuda' if torch.cuda.is_available() else 'cpu'
        self._cache_weights=None
        try:
            self.fe=AutoFeatureExtractor.from_pretrained(cfg['repo'],trust_remote_code=False)
            self.model=AutoModelForAudioClassification.from_pretrained(cfg['repo'],trust_remote_code=False)
        except Exception as exc:
            print('HF download problem; HTTPS identity fallback:',str(exc)[:180])
            folder=MODEL_CACHE/name
            for f in ['config.json','preprocessor_config.json','model.safetensors']:
                download_identity(cfg['repo'],f,folder/f,50000 if f.endswith('safetensors') else 100)
            self._cache_weights=folder/'model.safetensors'
            self.fe=AutoFeatureExtractor.from_pretrained(str(folder),local_files_only=True,trust_remote_code=False)
            self.model=AutoModelForAudioClassification.from_pretrained(str(folder),local_files_only=True,trust_remote_code=False)
        self.fake_idx,self.labels=fake_index_from_labels(self.model.config.id2label,cfg['expected_fake_index'])
        commit=getattr(self.model.config,'_commit_hash',None)
        self.weights_sha256,self.weights_file=locate_hf_weights(cfg['repo'],commit,self._cache_weights)
        self.model_revision=self.weights_sha256
        self.model=self.model.to(self.device).eval()
        self.model.requires_grad_(False)
        self.score_kind='softmax_fake_probability'
    def score_fake(self,w):
        batch=self.fe(w,sampling_rate=SR,return_tensors='pt',padding=True)
        batch={k:v.to(self.device) for k,v in batch.items()}
        with self.torch.inference_mode():
            probs=self.torch.softmax(self.model(**batch).logits,dim=-1)
        return float(probs[0,self.fake_idx].item())
    def close(self):
        del self.model
        if self.device=='cuda':self.torch.cuda.empty_cache()

def load_detector(name):
    cfg=MODEL_SOURCES[name]
    return OnnxAASIST(name,cfg) if cfg['kind']=='onnx' else HFWav2Vec2(name,cfg)

print('Обрано готові моделі:',SELECTED_MODELS)
print('Інференс без навчання: AASIST/AASIST-L через ONNX, Wav2Vec2 повторно НЕ запускається.')
# Батчинг тільки для HF Wav2Vec2: модель без жодного train() / gradient update.
def _wav2vec2_score_many(self,waves):
    tensors=self.fe([np.asarray(w,dtype=np.float32) for w in waves],sampling_rate=SR,return_tensors='pt',padding=True)
    tensors={k:v.to(self.device) for k,v in tensors.items()}
    with self.torch.inference_mode():
        probs=self.torch.softmax(self.model(**tensors).logits,dim=-1)
    return [float(x) for x in probs[:,self.fake_idx].detach().cpu().numpy()]
HFWav2Vec2.score_fake_many=_wav2vec2_score_many
# Перевірка апаратного забезпечення і споживання пам'яті
try:
 import torch
 print('CUDA available:',torch.cuda.is_available())
except Exception as e:print('Torch unavailable:',str(e)[:200])

In [ ]:
# Локальні незалежні тести індексації padding на коротких/повних сигналах.
x=np.arange(1,129,dtype=np.float32)/128
for n_active in (32,120,64000):
    q=np.pad(np.resize(x,n_active),(0,64000-n_active)).astype(np.float32)
    ref=np.tile(q[:n_active],int(np.ceil(64600/n_active)))[:64600]
    assert len(ref)==64600 and np.array_equal(ref[:n_active],q[:n_active])
    assert ref[-1]==q[(64599%n_active)]
assert not np.array_equal(np.pad(q,(0,600)),np.tile(q,int(np.ceil(64600/64000)))[:64600])
for name in ('AASIST_ONNX','AASIST_L_ONNX'):
    fp=MODEL_CACHE/MODEL_SOURCES[name]['file']
    assert fp.exists() and sha256_file(fp)==OLD_PROVENANCE[name]['weights_sha256'],f'ONNX SHA змінилася: {name}'
print('PASS: repeat padding / checkpoint SHA / hidden protected')

## 7. Новий протокол і пороги: калібрування з усіх 110 TRAIN-пар

In [ ]:
calibration=matched_df.loc[matched_df.cohort.eq('MATCHED_TRAIN')].copy()
assert not calibration.empty and set(calibration.label)=={0,1}
assert calibration.groupby('pair_id').label.nunique().eq(2).all()
assert set(calibration.pair_id).isdisjoint(set(matched_df.loc[matched_df.cohort.eq('MATCHED_VALIDATION'),'pair_id']))
# Не обмежуємо calibration до 48: використовуємо ВСІ наявні train-пари.
print('Порогова калібровка: ВСІ',calibration.pair_id.nunique(),'train-пар (без навчання ваг).')

def choose_threshold(y,scores):
    y=np.asarray(y,int);scores=np.asarray(scores,float)
    assert set(y)=={0,1} and np.isfinite(scores).all()
    vals=np.unique(scores)
    if len(vals)==1:return float(vals[0]),0.5
    candidates=np.r_[np.nextafter(vals[0],-np.inf), (vals[1:]+vals[:-1])/2, np.nextafter(vals[-1],np.inf)]
    # Модельний поріг за train без адаптації ваг
    results=np.asarray([balanced_accuracy_score(y,scores>=v) for v in candidates])
    ix=np.flatnonzero(np.isclose(results,results.max()))
    return float(candidates[ix[len(ix)//2]]),float(results.max())

CONFIG={'protocol':PROTOCOL_ID,'dataset_sha':sha256_file(MANIFEST_PATH),'catalog_sha':sha256_file(catalog_dir/'01_catalog_all.csv'),
  'voices_excluded_for_hidden':sorted(hidden_sha),'scenario_hash':SCENARIO_HASH,
  'model_names':[m for m,v in MODEL_ENABLED.items() if v],
  'segment_sec':WINDOW_SEC,'min_tail_sec':MIN_TAIL_SEC,'all_segments':ALL_NONOVERLAPPING_SEGMENTS,
  'n_public_matched_segments':len(matched_df),'n_aua_segments':len(aua_df),'n_total_segments':len(work),
  'bootstrap_reps':BOOTSTRAP_REPS,'seed':SEED,'cache_mp3':CACHE_MP3_TRANSFORMS,
  'threshold_policy':'ALL matched train pairs, no model updates',
  'preprocessing':PREPROCESSING_ID,'old_full_protocol_sha':sha256_file(OLD_OUT/'00_COMPLETE_PROTOCOL.json'),
  'wav2vec2_reused_from_old_v9':True}
CONF_FILE=OUT/'00_COMPLETE_PROTOCOL.json'
if CONF_FILE.exists():
    assert json.loads(CONF_FILE.read_text(encoding='utf8'))==CONFIG,'Конфігурацію змінено. Не змішуйте кеші: скопіюйте до іншого OUT.'
else:write_json_atomic(CONF_FILE,CONFIG)
RUN_HASH=sha256_file(CONF_FILE)
CAL_FILE=OUT/'04_CALIBRATION.json';PROV_FILE=OUT/'03_MODEL_PROVENANCE.json'
cal_meta=json.loads(CAL_FILE.read_text()) if CAL_FILE.exists() else {}
model_provenance=json.loads(PROV_FILE.read_text()) if PROV_FILE.exists() else {}
print('Зафіксовано повний протокол:',RUN_HASH)

## 8. Відновлення результатів Wav2Vec2 із v9 (без обчислень і без змін)

In [ ]:
# Крос-перевірка незмінності самого корпусу/сценаріїв відносно v9.
for key in ('dataset_sha','catalog_sha','scenario_hash','n_public_matched_segments',
            'n_aua_segments','n_total_segments','segment_sec','all_segments'):
    assert CONFIG[key]==OLD_CONFIG[key],f'Матриця змінилася після v9: {key}'
assert sha256_file(OUT/'02_SCENARIO_PROTOCOL.csv')==sha256_file(OLD_OUT/'02_SCENARIO_PROTOCOL.csv')
assert not (OLD_BASE/'HIDDEN_ONCE_STARTED_v9.json').exists()
assert len(work)==13910 and len(SCENARIOS)==21, (len(work),len(SCENARIOS))

DB_FILE=OUT/'05_FULL_PREDICTIONS.sqlite3'
conn=sqlite3.connect(DB_FILE,timeout=60)
conn.execute('PRAGMA journal_mode=WAL'); conn.execute('PRAGMA busy_timeout=60000')
conn.execute('''CREATE TABLE IF NOT EXISTS predictions (
 model TEXT NOT NULL, cohort TEXT NOT NULL,sample_id TEXT NOT NULL,scenario TEXT NOT NULL,
 record_id TEXT NOT NULL,source TEXT NOT NULL,split TEXT NOT NULL,pair_id TEXT NOT NULL,
 label INTEGER NOT NULL,voice_group TEXT,score_fake REAL NOT NULL,threshold REAL NOT NULL,
 pred_label INTEGER NOT NULL,checkpoint_sha TEXT NOT NULL,record_sha TEXT,
 family TEXT NOT NULL,level INTEGER NOT NULL,
 PRIMARY KEY(model,cohort,sample_id,scenario))''')
conn.commit()

assert REUSE_V9_WAV2VEC2
WAV_NAME='WAV2VEC2_STAFFORD'
OLD_WAV_PROV=OLD_PROVENANCE[WAV_NAME]
OLD_WAV_CAL=OLD_CALIBRATION[WAV_NAME]
assert OLD_WAV_PROV['weights_sha256']==OLD_WAV_CAL['weights_sha256']
assert OLD_WAV_CAL['protocol_sha']==OLD_READINESS['protocol_sha']
assert OLD_WAV_CAL['calibration_pairs']==110
assert OLD_READINESS['model_sha'][WAV_NAME]==OLD_WAV_PROV['weights_sha256']
assert OLD_CONFIG['model_names']==['AASIST_ONNX','AASIST_L_ONNX','WAV2VEC2_STAFFORD']

old_db=OLD_OUT/'05_FULL_PREDICTIONS.sqlite3'
old_csv=OLD_OUT/'05_ALL_SEGMENT_PREDICTIONS.csv'
work_ids=set(work.sample_id.astype(str))
if old_db.exists():
    ro=sqlite3.connect('file:'+str(old_db)+'?mode=ro',uri=True,timeout=30)
    prior_n=ro.execute('SELECT COUNT(*) FROM predictions WHERE model=?',(WAV_NAME,)).fetchone()[0]
    assert prior_n==len(work)*len(SCENARIOS),(prior_n,len(work)*21)
    prior_ids={r[0] for r in ro.execute('SELECT DISTINCT sample_id FROM predictions WHERE model=?',(WAV_NAME,))}
    assert prior_ids==work_ids,'Ідентифікатори аудіосегментів змінилися'
    prior_checks=ro.execute('''SELECT MIN(checkpoint_sha),MAX(checkpoint_sha),MIN(threshold),MAX(threshold)
                              FROM predictions WHERE model=?''',(WAV_NAME,)).fetchone()
    assert prior_checks[0]==prior_checks[1]==OLD_WAV_PROV['weights_sha256']
    assert np.isclose(prior_checks[2],prior_checks[3]) and np.isclose(prior_checks[2],OLD_WAV_CAL['threshold'])
    cur=ro.execute('''SELECT model,cohort,sample_id,scenario,record_id,source,split,pair_id,
                     label,voice_group,score_fake,threshold,pred_label,checkpoint_sha,record_sha,family,level
                     FROM predictions WHERE model=?''',(WAV_NAME,))
    total_read=0
    while True:
        batch=cur.fetchmany(5000)
        if not batch:break
        with conn:conn.executemany('INSERT OR IGNORE INTO predictions VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',batch)
        total_read+=len(batch)
    ro.close()
    print('Wav2Vec2: перенесено з v9 SQLite',total_read,'прогнозів')
else:
    assert old_csv.exists(),'Не знайдено ні v9 SQLite, ні v9 CSV із прогнозами Wav2Vec2'
    cols=['model','cohort','sample_id','scenario','record_id','source','split','pair_id','label','voice_group',
          'score_fake','threshold','pred_label','checkpoint_sha','record_sha','family','level']
    read_total=0; seen_ids=set()
    for chunk in pd.read_csv(old_csv,chunksize=40_000,low_memory=False,keep_default_na=False):
        sub=chunk.loc[chunk.model.eq(WAV_NAME)].copy()
        if sub.empty:continue
        sub['record_sha']=''
        assert all(s in sub.columns for s in cols)
        seen_ids.update(sub.sample_id.astype(str))
        with conn:conn.executemany('INSERT OR IGNORE INTO predictions VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
             list(sub[cols].itertuples(index=False,name=None)))
        read_total+=len(sub)
    assert read_total==len(work)*21 and seen_ids==work_ids
    print('Wav2Vec2: перенесено з v9 CSV',read_total,'прогнозів')

count=conn.execute('SELECT COUNT(*) FROM predictions WHERE model=?',(WAV_NAME,)).fetchone()[0]
assert count==len(work)*21, ('Неповний кеш Wav2Vec2:',count)
assert {r[0] for r in conn.execute('SELECT DISTINCT scenario FROM predictions WHERE model=?',(WAV_NAME,))}==set(SC.scenario)
model_provenance[WAV_NAME]=OLD_WAV_PROV
cal_meta[WAV_NAME]=OLD_WAV_CAL
write_json_atomic(PROV_FILE,model_provenance)
write_json_atomic(CAL_FILE,cal_meta)
write_json_atomic(OUT/'00_WAV2VEC2_REUSE_AUDIT.json',{
    'reused_model':WAV_NAME,'source_v9_dir':str(OLD_OUT),'prediction_count':count,
    'checkpoint_sha256':OLD_WAV_PROV['weights_sha256'], 'calibration':OLD_WAV_CAL,
    'preprocessing_unchanged':True,'fresh_wav2vec2_inferences':0,
    'hidden_opened':False})
print('PASS: Wav2Vec2 збережено без змін; у новому каталозі',count,'прогнозів')

## 9. Повний AASIST/AASIST-L інференс із відновленням після переривання

In [ ]:
# SHA кожного початкового запису обчислюємо лише один раз, але регулярно перевіряємо.
verified_hashes={}
errors=[]

def verify_record(row):
    key=str(row.path)
    if key in verified_hashes:return verified_hashes[key]
    if not Path(key).is_file():raise FileNotFoundError(key)
    sha=sha256_file(key) if VERIFY_SHA256 else ''
    expected=str(row.expected_sha).strip().lower()
    if VERIFY_SHA256 and len(expected)==64 and expected!=sha:
        raise RuntimeError(f'Не збігається SHA-256 {row.cohort}/{row.sample_id}: {key}')
    verified_hashes[key]=sha
    return sha

def check_completed(model,row):
    num=conn.execute('SELECT COUNT(*) FROM predictions WHERE model=? AND cohort=? AND sample_id=?',
                    (model,row.cohort,row.sample_id)).fetchone()[0]
    if num not in (0,len(SCENARIOS)):raise RuntimeError('Частково записаний сегмент: '+str((model,row.sample_id,num)))
    return num==len(SCENARIOS)

def evaluate_all_for_model(name,rows):
    detector=load_detector(name)
    try:
        digest=detector.weights_sha256
        assert len(digest)==64
        prov={'source':detector.source,'weights_sha256':digest,'score_kind':detector.score_kind,
              'model_revision':detector.model_revision,'preprocessing':PREPROCESSING_ID,'id2label':{str(k):str(v) for k,v in detector.labels.items()},'training_performed':False}
        assert digest==OLD_PROVENANCE[name]['weights_sha256'],'Ваги AASIST відрізняються від v9'
        if name in model_provenance:
            assert model_provenance[name]==prov,'Змінилася версія pretrained-моделі або орієнтація score'
        model_provenance[name]=prov;write_json_atomic(PROV_FILE,model_provenance)
        if name in cal_meta:
            cal=cal_meta[name]
            assert cal['weights_sha256']==digest and cal['protocol_sha']==RUN_HASH
            threshold=float(cal['threshold'])
        else:
            y=[];scores=[]
            for r in calibration.itertuples(index=False):
                verify_record(r)
                wave=read_wave(r.path,r.start_sec,r.active_n)
                scores.append(detector.score_fake(wave,active_n=r.active_n));y.append(r.label)
            threshold,train_ba=choose_threshold(y,scores)
            train_auc=float(roc_auc_score(y,scores))
            cal_meta[name]={'weights_sha256':digest,'threshold':threshold,'calibration_pairs':int(calibration.pair_id.nunique()),
                            'train_BA':train_ba,'train_AUC':train_auc,'protocol_sha':RUN_HASH,
                            'preprocessing':PREPROCESSING_ID,'no_model_training':True}
            write_json_atomic(CAL_FILE,cal_meta)
        print('MODEL:',name,'threshold=',round(threshold,5),'train_AUC=',round(cal_meta[name]['train_AUC'],4))
        if cal_meta[name]['train_AUC']<=0.5:
            print('УВАГА: AUC train <= 0.5: потрібна перевірка орієнтації score, автоматично знак НЕ інвертуємо')
        nnew=0;nskip=0
        batch=[];tick=time.time()
        for ix,r in enumerate(rows.itertuples(index=False),1):
            if check_completed(name,r):nskip+=1;continue
            file_sha=verify_record(r)
            w=read_wave(r.path,r.start_sec,r.active_n)
            waves=[transform_cached(w,sc,r,OUT/'_mp3_cache') for sc in SCENARIOS]
            if hasattr(detector,'score_fake_many'):
                scores=[]
                for off in range(0,len(waves),4):
                    scores.extend(detector.score_fake_many(waves[off:off+4]))
            else:
                scores=[detector.score_fake(z,active_n=r.active_n) for z in waves]
            assert len(scores)==len(SCENARIOS)
            for sc,p in zip(SCENARIOS,scores):
                score=float(p)
                assert np.isfinite(score),(name,r.sample_id,sc['scenario'])
                batch.append((name,r.cohort,r.sample_id,sc['scenario'],r.record_id,r.source,r.split,str(r.pair_id),
                              int(r.label),str(r.group_id),score,float(threshold),int(score>=threshold),digest,file_sha,
                              sc['family'],int(sc['level'])))
            nnew+=len(SCENARIOS)
            if ix%25==0 or len(batch)>=1000:
                with conn:conn.executemany('INSERT OR IGNORE INTO predictions VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',batch)
                batch=[]
            if ix%250==0:
                print(f'{name}: {ix:,}/{len(rows):,} аудіосегментів; нових прогнозів {nnew:,}; у кеші {nskip:,}; минуло {time.time()-tick:.0f} с')
        if batch:
            with conn:conn.executemany('INSERT OR IGNORE INTO predictions VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',batch)
        print(name,'DONE. Нових прогнозів:',nnew,', уже в кеші сегментів:',nskip)
    finally:
        detector.close();del detector;gc.collect()

if RUN_PHASE=='FULL':
    for model in ('AASIST_ONNX','AASIST_L_ONNX'):evaluate_all_for_model(model,work)
else:print('Фаза',RUN_PHASE,': жодного нового інференсу у відкритих даних; перевіряємо попередній FULL.')

expected=len(work)*len(SCENARIOS)*len(SELECTED_MODELS)
actual=conn.execute('SELECT COUNT(*) FROM predictions').fetchone()[0]
assert actual==expected,(f'Неповна матриця: {actual:,}/{expected:,}. У FULL — повторний Run All продовжить з місця зупинки.')
for model in SELECTED_MODELS:
    group=conn.execute('SELECT COUNT(*),MIN(checkpoint_sha),MAX(checkpoint_sha),MIN(threshold),MAX(threshold) FROM predictions WHERE model=?',(model,)).fetchone()
    assert group[0]==len(work)*21 and group[1]==group[2]==model_provenance[model]['weights_sha256']
    assert np.isclose(group[3],group[4]) and np.isclose(group[3],cal_meta[model]['threshold'])
print(f'ПОКРИТТЯ 100%: {actual:,}/{expected:,} прогнозів, 3 pretrained-моделі × 21 сценарій × ВСІ сегменти')

## 10. Агрегація до рівня записів, метрики, аналіз джерел

In [ ]:
pred=pd.read_sql_query('SELECT * FROM predictions',conn)
assert len(pred)==expected
# Для оригінального AUA усі сегменти одного джерельного запису агрегуємо за score;
# НЕ рахуємо сегменти різних вікон за незалежних носіїв.
record=(pred.groupby(['model','cohort','record_id','source','split','pair_id','label','scenario','family','level'],as_index=False)
      .agg(score_fake=('score_fake','mean'),threshold=('threshold','first'),n_segments=('sample_id','nunique')))
record['pred_label']=(record.score_fake>=record.threshold).astype(int)
assert record.groupby(['model','cohort','record_id']).scenario.nunique().eq(21).all()
record.to_csv(OUT/'05_RECORD_LEVEL_PREDICTIONS.csv',index=False)
# Експорт великих сирих прогнозів для повної перевірки; лише псевдонімізовані ID, ніяких WAV.
pred.drop(columns=['record_sha'],errors='ignore').to_csv(OUT/'05_ALL_SEGMENT_PREDICTIONS.csv',index=False)

# Класи можуть бути в AUA прив'язані до джерела. Тому спільні BA/AUC AUA — лише конфундовані.
def eer_score(y,score):
    fpr,tpr,_=roc_curve(y,score)
    fnr=1-tpr;delta=fpr-fnr
    cross=np.flatnonzero(delta[:-1]*delta[1:]<=0)
    if len(cross):
        i=int(cross[0]);d=delta[i+1]-delta[i]
        v=0 if abs(d)<1e-15 else -delta[i]/d
        return float(fpr[i]+v*(fpr[i+1]-fpr[i]))
    return float((fpr[np.argmin(abs(delta))]+fnr[np.argmin(abs(delta))])/2)

def binary_metrics(df):
    y=df.label.to_numpy(int);p=df.pred_label.to_numpy(int);scores=df.score_fake.to_numpy(float)
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    ans={'n_records':len(df),'n_real':int((y==0).sum()),'n_fake':int((y==1).sum()),
         'TP':int(tp),'TN':int(tn),'FP':int(fp),'FN':int(fn),
         'TPR_fake':float(tp/(tp+fn)) if tp+fn else np.nan,
         'TNR_real':float(tn/(tn+fp)) if tn+fp else np.nan,
         'F1_fake':float(f1_score(y,p,zero_division=0)) if (y==1).any() else np.nan}
    if set(y)=={0,1}:
        ans.update(BA=float(balanced_accuracy_score(y,p)),ROC_AUC=float(roc_auc_score(y,scores)),EER=float(eer_score(y,scores)))
    else:ans.update(BA=np.nan,ROC_AUC=np.nan,EER=np.nan)
    return ans

outrows=[]
for (m,coh,s,f,l),g in record.groupby(['model','cohort','scenario','family','level'],sort=True):
    q=binary_metrics(g);q.update(model=m,cohort=coh,scenario=s,family=f,level=int(l),
       inferential_status='paired_validation' if coh=='MATCHED_VALIDATION' else ('CALIBRATION_IN_SAMPLE' if coh=='MATCHED_TRAIN' else ('SOURCE_LABEL_CONFOUNDED' if coh=='AUA_CANONICAL' else 'EXPLORATORY_ADAPTATION')))
    outrows.append(q)
metrics=pd.DataFrame(outrows)
metrics.to_csv(OUT/'06_ALL_SCENARIO_METRICS.csv',index=False)

source_rows=[]
for (m,src,s,f,l,y),g in record.loc[record.cohort.eq('AUA_CANONICAL')].groupby(['model','source','scenario','family','level','label']):
    vals=binary_metrics(g)
    source_rows.append({'model':m,'source':src,'scenario':s,'family':f,'level':int(l),'label':int(y),**vals})
source_metrics=pd.DataFrame(source_rows)
source_metrics.to_csv(OUT/'06_AUA_SOURCE_SPECIFIC_ERROR_PROFILES.csv',index=False)

summary=[]
for (coh,model),g in metrics.groupby(['cohort','model']):
    clean=g.loc[g.family.eq('clean')].iloc[0]
    cats=g.loc[g.family.ne('clean')].groupby('family').BA.mean()
    asri=float(cats.mean()) if cats.notna().all() and len(cats)==4 else np.nan
    summary.append({'cohort':coh,'model':model,'n_original_recordings':int(clean.n_records),
                    'clean_BA':float(clean.BA),'ASRI_Audio':asri,
                    'worst_stress_BA':float(g.loc[g.family.ne('clean'),'BA'].min()),
                    'clean_ROC_AUC':float(clean.ROC_AUC),
                    'scope':'NON-INDEPENDENT SOURCE/CLASS CONFOUNDING' if coh=='AUA_CANONICAL' else
                       ('PRIMARY INDEPENDENT VALIDATION; single TTS voice' if coh=='MATCHED_VALIDATION' else 'DESCRIPTIVE / calibration or adaptation')})
summary=pd.DataFrame(summary).sort_values(['cohort','ASRI_Audio'],ascending=[True,False])
summary.to_csv(OUT/'06_COMPLETED_SUMMARY.csv',index=False)
print('Основна validation (НЕ змішуємо з AUA або train):')
display(summary.loc[summary.cohort.eq('MATCHED_VALIDATION')])
print('Повна матриця джерел AUA:')
display(source_metrics.groupby(['source','label']).n_records.max().reset_index())

## 11. Валідаційна статистика й багатокритеріальне ранжування

In [ ]:
matched_val=record.loc[record.cohort.eq('MATCHED_VALIDATION')].copy()
assert set(matched_val.label)=={0,1}
pairs=np.asarray(sorted(matched_val.pair_id.astype(str).unique()))
assert len(pairs)==raw.loc[raw.split.eq('validation')].pair_id.nunique(),'Пара не пройшла FULL VALIDATION'
model_list=SELECTED_MODELS
# Формуємо повну матрицю для парного bootstrap зі ЗБЕРЕЖЕННЯМ кореляції scenario × model.
vec={}
for (m,s),g in matched_val.groupby(['model','scenario']):
    piv=g.pivot(index='pair_id',columns='label',values='pred_label').reindex(pairs)
    assert not piv.isna().any().any(),(m,s)
    vec[(m,s)]=piv[[0,1]].to_numpy(np.int8)
FAMILIES=['noise','mp3','reverb','lowpass']

def asri_sample(model,ids,family=None):
    flist=[family] if family else FAMILIES
    values=[]
    for fam in flist:
        stages=[sc['scenario'] for sc in SCENARIOS if sc['family']==fam]
        each=[]
        for name in stages:
            pp=vec[(model,name)][ids]
            each.append(.5*(np.mean(pp[:,0]==0)+np.mean(pp[:,1]==1)))
        values.append(float(np.mean(each)))
    return float(np.mean(values))

rng=np.random.default_rng(SEED+21)
N=len(pairs)
boots={m:np.zeros(BOOTSTRAP_REPS,dtype=float) for m in model_list}
for k in range(BOOTSTRAP_REPS):
    ids=rng.integers(0,N,size=N)
    for m in model_list:boots[m][k]=asri_sample(m,ids)
ci=[]
for m in model_list:
    xx=boots[m]
    ci.append(dict(model=m,mean_ASRI=asri_sample(m,np.arange(N)),
                   ci_lower=float(np.quantile(xx,.025)),ci_upper=float(np.quantile(xx,.975)),
                   CI_SCOPE='conditional on one known TTS voice; pair bootstrap, NOT generator CI'))
ci=pd.DataFrame(ci);ci.to_csv(OUT/'07_MATCHED_VALIDATION_BOOTSTRAP_CI.csv',index=False)
contrasts=[]
for a,b in itertools.combinations(model_list,2):
    d=boots[a]-boots[b]
    contrasts.append({'model_a':a,'model_b':b,'difference_ASRI':float(ci.set_index('model').loc[a,'mean_ASRI']-ci.set_index('model').loc[b,'mean_ASRI']),
                      'ci_lower':float(np.quantile(d,.025)),'ci_upper':float(np.quantile(d,.975)),
                      'CI_excludes_zero':bool(np.quantile(d,.025)>0 or np.quantile(d,.975)<0)})
contrast=pd.DataFrame(contrasts);contrast.to_csv(OUT/'07_PAIRWISE_MODEL_ASRI_DIFFERENCES.csv',index=False)

# Додатково кластерний bootstrap за справжніми ЛЮДСЬКИМИ мовцями в validation.
# Один fake TTS-голос означає, що і цей CI лишається conditional on that voice.
valreal=raw.loc[(raw.split.eq('validation')) & (raw.label.eq(0)),['pair_id','group_id']].drop_duplicates()
valreal['pair_id']=valreal.pair_id.astype(str)
assert valreal.pair_id.nunique()==N
speaker_groups={sp:np.asarray([np.where(pairs==pid)[0][0] for pid in g.pair_id],dtype=int)
                for sp,g in valreal.groupby('group_id')}
assert len(speaker_groups)>0
blocks=list(speaker_groups.values())
speaker_boot={m:np.zeros(BOOTSTRAP_REPS,dtype=float) for m in model_list}
for k in range(BOOTSTRAP_REPS):
    chosen=rng.integers(0,len(blocks),size=len(blocks))
    ids=np.concatenate([blocks[j] for j in chosen])
    for m in model_list:speaker_boot[m][k]=asri_sample(m,ids)
speaker_ci=pd.DataFrame([
    {'model':m,'ASRI_Audio':asri_sample(m,np.arange(N)),'CI_lower':float(np.quantile(speaker_boot[m],.025)),
     'CI_upper':float(np.quantile(speaker_boot[m],.975)),
     'n_unique_real_speakers':int(len(blocks)),'n_synthetic_voices':1,
     'scope':'conditional on observed synthetic voice; speaker-cluster bootstrap'} for m in model_list])
speaker_ci.to_csv(OUT/'07_SPEAKER_CLUSTER_BOOTSTRAP_CI.csv',index=False)

# Додатковий парний Friedman test у 4 категоріях, але висновок умовний для TTS-голосу.
friedman=[]
for fam in FAMILIES:
    by_m=[]
    for m in model_list:
        fnames=[s['scenario'] for s in SCENARIOS if s['family']==fam]
        perpair=np.mean([.5*((vec[(m,s)][:,0]==0).astype(float)+(vec[(m,s)][:,1]==1).astype(float)) for s in fnames],axis=0)
        by_m.append(perpair)
    stat,pval=stats.friedmanchisquare(*by_m)
    friedman.append({'family':fam,'statistic':float(stat) if np.isfinite(stat) else np.nan,
                     'p_value':float(pval) if np.isfinite(pval) else np.nan,
                     'status':'OK' if np.isfinite(pval) else 'UNDEFINED_IDENTICAL_MODEL_RANKS',
                     'n_pairs':N,'scope':'conditional, repeated-measures pairs; one TTS voice'})
pd.DataFrame(friedman).to_csv(OUT/'07_MATCHED_VALIDATION_FRIEDMAN.csv',index=False)

# Адаптивна пріоритизація РИЗИКІВ, а не навчання: найгірші категорії та сценарії для кожної моделі.
valmetrics=metrics.loc[metrics.cohort.eq('MATCHED_VALIDATION')].copy()
valclean=valmetrics.loc[valmetrics.scenario.eq('clean')].set_index('model')
valmetrics['delta_BA_vs_clean']=valmetrics.apply(lambda r: r.BA-valclean.loc[r.model,'BA'],axis=1)
valmetrics['delta_AUC_vs_clean']=valmetrics.apply(lambda r:r.ROC_AUC-valclean.loc[r.model,'ROC_AUC'],axis=1)
risk=valmetrics.loc[valmetrics.family.ne('clean')].sort_values(['model','delta_BA_vs_clean','delta_AUC_vs_clean'])
risk['risk_rank']=risk.groupby('model').cumcount()+1
risk.to_csv(OUT/'08_SCENARIO_ADAPTIVE_RISK_PRIORITY.csv',index=False)
# Перехід лідера clean → stress: підсумки є описовими (CI вирішує значущість).
cleanrank=valclean.BA.rank(ascending=False,method='average')
stressrank=ci.set_index('model').mean_ASRI.rank(ascending=False,method='average')
ranks=pd.DataFrame({'model':model_list,'clean_rank':cleanrank.reindex(model_list).values,
                    'stress_rank':stressrank.reindex(model_list).values})
ranks['rank_change']=ranks.stress_rank-ranks.clean_rank
ranks.to_csv(OUT/'08_MODEL_RANK_CHANGES.csv',index=False)
# Мультикритеріальне оцінювання з різними критеріями, без випадкового повторного підрахунку.
crit=valmetrics.groupby('model').agg(mean_AUC=('ROC_AUC','mean'),min_BA=('BA','min'))
crit['ASRI']=ci.set_index('model').mean_ASRI
crit['clean_BA']=valclean.BA
norm=(crit-crit.min())/(crit.max()-crit.min()).replace(0,np.nan)
norm=norm.fillna(.5)
crit['MCDM_equal_weight_score']=norm.mean(axis=1)
crit['MCDM_rank']=crit.MCDM_equal_weight_score.rank(ascending=False,method='average')
crit.to_csv(OUT/'08_MCDM_EXPLORATORY.csv')

# Факторні моделі: не показуємо статистичної значущості, якщо незалежних голосів не вистачає.
voice_ids=raw.loc[(raw.split=='validation')&(raw.label==1),'tts_voice_id'].dropna().nunique()
stats_scope={'validation_pairs':int(N),'validation_fake_voice_ids':int(voice_ids),
 'bootstrap_unit':'REAL–TTS pair_id','CI_conditions':'within observed TTS voice(s), not across generators',
 'cross_generator_inference_supported':False if voice_ids<3 else 'not independently validated',
 'AUA_source_label_confounding':True,'model_training':False,'hidden_opened':False}
write_json_atomic(OUT/'07_STATISTICAL_SCOPE.json',stats_scope)
display(ci)
display(contrast)

## 12. Графіки сценаріїв

In [ ]:
heat=valmetrics.pivot(index='scenario',columns='model',values='BA').reindex([s['scenario'] for s in SCENARIOS])
fig,ax=plt.subplots(figsize=(10,11));im=ax.imshow(heat.to_numpy(float),vmin=0,vmax=1,aspect='auto')
ax.set_yticks(range(len(heat)),heat.index);ax.set_xticks(range(len(heat.columns)),heat.columns,rotation=20,ha='right')
ax.set_title(f'21 сценарій · matched validation · N={N} пар')
fig.colorbar(im,ax=ax,label='Balanced Accuracy');fig.tight_layout();fig.savefig(OUT/'09_MATCHED_VALIDATION_HEATMAP.png',dpi=170);plt.show();plt.close(fig)

fig,ax=plt.subplots(figsize=(9,5))
order=ci.model.tolist();x=np.arange(len(order));yy=ci.mean_ASRI.to_numpy()
err=np.vstack([yy-ci.ci_lower.to_numpy(),ci.ci_upper.to_numpy()-yy])
ax.errorbar(x,yy,yerr=np.maximum(err,0),fmt='o',capsize=5,label='ASRI-Audio, within-voice pair bootstrap')
ax.scatter(x,valclean.reindex(order).BA,marker='x',label='Clean BA')
ax.set_xticks(x,order,rotation=15);ax.set_ylim(0,1);ax.grid(axis='y',alpha=.25);ax.legend()
ax.set_title('Clean vs scenario robustness; 95% pair-bootstrap CI');fig.tight_layout()
fig.savefig(OUT/'09_ASRI_BOOTSTRAP_COMPARISON.png',dpi=170);plt.show();plt.close(fig)

for family in FAMILIES:
    g=valmetrics.loc[valmetrics.family.eq(family)]
    fig,ax=plt.subplots(figsize=(9,4.5))
    for m,t in g.groupby('model'):
        ax.plot(t.sort_values('level').level,t.sort_values('level').BA,marker='o',label=m)
    ax.set_ylim(0,1);ax.set_xticks([1,2,3,4,5]);ax.grid(alpha=.2)
    ax.set_title('Balanced Accuracy · '+family);ax.set_xlabel('Severity level');ax.set_ylabel('BA')
    ax.legend();fig.tight_layout();fig.savefig(OUT/f'09_FAMILY_{family.upper()}.png',dpi=160);plt.show();plt.close(fig)

# AUA: TPR on fake-only source, TNR on real-only source — не фіктивний BA окремого джерела.
sourceclean=source_metrics.loc[source_metrics.scenario.eq('clean')].copy()
sourceclean['correct_class_recall']=np.where(sourceclean.label.eq(1),sourceclean.TPR_fake,sourceclean.TNR_real)
fig,ax=plt.subplots(figsize=(11,5))
tab=sourceclean.pivot_table(index='source',columns='model',values='correct_class_recall')
tab.plot(kind='bar',ax=ax);ax.set_ylim(0,1);ax.set_ylabel('Recall for the source class')
ax.set_title('AUA: clean accuracy conditional on originating source (source↔label confounded)')
fig.tight_layout();fig.savefig(OUT/'09_AUA_BY_SOURCE_RECALL.png',dpi=160);plt.show();plt.close(fig)

if SHOW_TRAIN_AUDIO_EXAMPLE:
    r=calibration.iloc[0]
    wave=read_wave(r.path,r.start_sec,r.active_n)
    print('Лише train-аудіо; пара',r.pair_id,'; мітка',r.label)
    display(Audio(wave,rate=SR))
    fig,ax=plt.subplots(figsize=(10,3));ax.plot(np.arange(len(wave))/SR,wave,lw=.4)
    ax.set_xlabel('Time [s]');ax.set_ylabel('Amplitude');ax.set_title('Train waveform');fig.tight_layout()
    fig.savefig(OUT/'09_TRAIN_AUDIO_WAVEFORM.png',dpi=160);plt.show();plt.close(fig)

## 13. Чітке порівняння v9 (zero-padding) ↔ v9.1 (repeat-padding)

In [ ]:
old_metrics=pd.read_csv(OLD_OUT/'06_ALL_SCENARIO_METRICS.csv',low_memory=False)
new_metrics=pd.read_csv(OUT/'06_ALL_SCENARIO_METRICS.csv',low_memory=False)
keys=['model','cohort','scenario','family','level']
old_sub=old_metrics[keys+['BA','ROC_AUC','TPR_fake','TNR_real']].rename(columns={k:'v9_'+k for k in ['BA','ROC_AUC','TPR_fake','TNR_real']})
new_sub=new_metrics[keys+['BA','ROC_AUC','TPR_fake','TNR_real']].rename(columns={k:'v9_1_'+k for k in ['BA','ROC_AUC','TPR_fake','TNR_real']})
comparison=old_sub.merge(new_sub,on=keys,validate='one_to_one')
assert len(comparison)==len(old_metrics)==len(new_metrics)
for col in ['BA','ROC_AUC','TPR_fake','TNR_real']:
    comparison['delta_'+col]=comparison['v9_1_'+col]-comparison['v9_'+col]
comparison.to_csv(OUT/'11_PREPROCESSING_EFFECT_BY_SCENARIO.csv',index=False)
# Wav2Vec2 має залишитися буквально ідентичним на кожній групі і на кожному сценарії.
wav_delta=comparison.loc[comparison.model.eq(WAV_NAME),['delta_BA','delta_ROC_AUC','delta_TPR_fake','delta_TNR_real']]
assert np.allclose(wav_delta.to_numpy(float),0,atol=1e-12,equal_nan=True),'Wav2Vec2 помилково змінено!'

s0=pd.read_csv(OLD_OUT/'06_COMPLETED_SUMMARY.csv')
s1=pd.read_csv(OUT/'06_COMPLETED_SUMMARY.csv')
summary_comp=s0.merge(s1,on=['cohort','model'],suffixes=('_v9','_v9_1'),validate='one_to_one')
for metric in ['clean_BA','ASRI_Audio','clean_ROC_AUC','worst_stress_BA']:
    summary_comp['delta_'+metric]=summary_comp[metric+'_v9_1']-summary_comp[metric+'_v9']
summary_comp.to_csv(OUT/'11_PREPROCESSING_EFFECT_SUMMARY.csv',index=False)

# Відсоток рішень, які змінилися: обчислюємо на незалежному record-level, не на вікнах.
old_rec=pd.read_csv(OLD_OUT/'05_RECORD_LEVEL_PREDICTIONS.csv',
    usecols=['model','cohort','record_id','scenario','pred_label'],low_memory=False)
new_rec=record[['model','cohort','record_id','scenario','pred_label']].copy()
key_rec=['model','cohort','record_id','scenario']
changed=old_rec.merge(new_rec,on=key_rec,suffixes=('_v9','_v9_1'),validate='one_to_one')
assert len(changed)==len(old_rec)==len(new_rec)
changed['decision_changed']=changed.pred_label_v9.ne(changed.pred_label_v9_1)
change_table=changed.groupby(['cohort','model']).agg(
    n_record_scenarios=('decision_changed','size'),
    n_changed=('decision_changed','sum'),
    fraction_changed=('decision_changed','mean')).reset_index()
assert change_table.loc[change_table.model.eq(WAV_NAME),'n_changed'].sum()==0
change_table.to_csv(OUT/'11_PREPROCESSING_CHANGED_DECISIONS.csv',index=False)
print('ЗМІНИ validation v9 → v9.1 (повний повтор 21 сценарію):')
display(summary_comp.loc[summary_comp.cohort.eq('MATCHED_VALIDATION'),
    ['model','clean_BA_v9','clean_BA_v9_1','ASRI_Audio_v9','ASRI_Audio_v9_1','delta_ASRI_Audio']])
display(change_table.loc[change_table.cohort.eq('MATCHED_VALIDATION')])

fig,ax=plt.subplots(figsize=(9,4.5))
a=summary_comp.loc[summary_comp.cohort.eq('MATCHED_VALIDATION')].set_index('model').reindex(SELECTED_MODELS)
xx=np.arange(len(a));width=.36
ax.bar(xx-width/2,a.ASRI_Audio_v9,width,label='v9: zero-padding')
ax.bar(xx+width/2,a.ASRI_Audio_v9_1,width,label='v9.1: repeat-padding')
ax.set_xticks(xx,a.index,rotation=10);ax.set_ylabel('ASRI-Audio (0–1)')
ax.set_ylim(0,1);ax.legend();ax.set_title('Matched validation: вплив доповнення сигналу')
fig.tight_layout();fig.savefig(OUT/'11_ZERO_VS_REPEAT_ASRI.png',dpi=160);plt.show();plt.close(fig)

## 14. Повний машинний звіт та ZIP — старі v9-файли не змінюються

In [ ]:
report={'run_phase':'FULL','protocol_sha':RUN_HASH,'model_training':False,
 'source_reference':'https://doi.org/10.3390/make8060161',
 'preprocessing':PREPROCESSING_ID,'previous_v9_dir':str(OLD_OUT),
 'wav2vec2_reused_without_inference':True,'hidden_test_still_closed':True,
 'canonical_catalog_files':int(len(catalog)),'canonical_usable_files':int(len(canonical)),
 'canonical_public_files':int(aua_df.record_id.nunique()),'canonical_windows':int(len(aua_df)),
 'matched_public_pairs':int(public.pair_id.nunique()),'matched_train_pairs':int(raw.loc[raw.split.eq('train')].pair_id.nunique()),
 'matched_adaptation_pairs':int(raw.loc[raw.split.eq('adaptation')].pair_id.nunique()),
 'matched_validation_pairs':int(raw.loc[raw.split.eq('validation')].pair_id.nunique()),
 'matched_hidden_pairs_protected':int(hidden.pair_id.nunique()),
 'hidden_overlap_original_excluded':int(len(excluded_hidden)),
 'n_models':len(SELECTED_MODELS),'n_scenarios':len(SCENARIOS),
 'planned_predictions':int(expected),'actual_predictions':int(actual),
 'model_sha':{m:model_provenance[m]['weights_sha256'] for m in SELECTED_MODELS},
 'full_unrestricted_experiment':True,'hidden_test_opened':False,
 'findings_status':'REPORT ONLY: do not interpret full AUA as independent ROC-AUC because source↔label confounding',
 'one_fake_voice_validation_warning':int(voice_ids)<2,
 'no_verified_cross_generator_generalization':True,
 'record_level_aggregation':True,'all_nonoverlap_segments':ALL_NONOVERLAPPING_SEGMENTS}
write_json_atomic(OUT/'10_FULL_EXPERIMENT_READINESS.json',report)
assert expected==actual and set(SELECTED_MODELS)==set(model_provenance)
# Кеш тимчасових MP3-масивів після 100% завершення не потрібний: усі прогнози збережені в SQLite.
cache_dir=OUT/'_mp3_cache'
if cache_dir.exists() and RUN_PHASE=='FULL':
    shutil.rmtree(cache_dir)
    print('Тимчасовий кеш MP3 очищено після повного завершення; джерельні WAV не змінено.')
ZIP=RESULTS_ROOT/'SAADE_AUA_v9_1_REPEAT_PADDING_FULL_FOR_REVIEW.zip'
keep={'.csv','.json','.txt','.png'}
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file() and f.suffix in keep and not f.name.startswith('01_PRIVATE_'):
            z.write(f,arcname=f.name)
    for name in ['01_sources_summary.csv','01_source_class_table.csv','01_data_diagnostics.json','01_DRSSU_PROVENANCE.json']:
        f=catalog_dir/name
        if f.exists():z.write(f,arcname='catalog_audit/'+name)
print('Повний звіт:',ZIP,'|',round(ZIP.stat().st_size/1e6,2),'MB')
print('v9.1 FULL завершено з repeat-padding для AASIST/AASIST-L; hidden ЗАКРИТО. Wav2Vec2 запозичено з v9.')

## 15. Дослідницька інтерпретація та межі

- **Метод:** 21 незмінний акустичний сценарій, усі придатні сегменти AUA та всі відкриті matched-пари; у двох моделях змінено **лише** ONNX padding перед інференсом.
- **Wav2Vec2:** результати повністю перенесено зі старого v9; моделі не навчаються.
- **Пороги:** AASIST/AASIST-L повторно відкалібровано **тільки на train** після нової передобробки; validation не використовується для калібрування.
- **Статистичне обмеження:** validation містить один синтетичний голос Piper (`tetiana`); міжгенераторні висновки залишаються недоведеними. AUA має конфундування `source ↔ label`.
- **Оприлюднення:** новий v9.1 — **post-hoc correction** після перегляду validation v9; зміну необхідно задокументувати в статті. Не змішуйте результати старого v9 із новими як один протокол.
- **Hidden:** у цьому ноутбуці немає коду його відкриття. Після перевірки v9.1 окремо фіксуємо новий протокол та проводимо одноразову фінальну перевірку.